# Unidad 4 · Colab 2 de 3
## Desarrollo de APIs RESTful con FastAPI/Flask y Pydantic

**Objetivos de este notebook**

- Crear endpoints RESTful con **FastAPI**.
- Definir modelos de datos con **Pydantic** para validación automática y serialización.
- Entender la documentación interactiva automática (Swagger UI / OpenAPI).
- Comparar brevemente con **Flask**, la alternativa más liviana y menos opinionada.

> **Nivel:** intermedio. Se asume que ya consumiste APIs (Colab 1) y sabés Python con type hints básicos.

**Cómo probamos la API en este notebook:** usamos `TestClient` de FastAPI, que simula requests HTTP sin necesitar levantar un servidor real — ideal para Colab. Al final te mostramos cómo correrla de verdad con `uvicorn`.

---

## 1. FastAPI vs. Flask

| | FastAPI | Flask |
|---|---|---|
| Validación de datos | Automática, vía Pydantic | Manual (o con extensiones como flask-pydantic) |
| Documentación | Automática (Swagger UI / ReDoc) | Manual o con extensiones |
| Async nativo | Sí | Limitado (mejora en versiones recientes) |
| Curva de aprendizaje | Un poco más de magia (type hints) | Más minimalista y explícito |
| Cuándo usarlo | APIs nuevas, con muchos datos estructurados | Proyectos simples, o con mucho código legado |

Documentación oficial: [FastAPI](https://fastapi.tiangolo.com/) · [Flask](https://flask.palletsprojects.com/)

## 2. Primeros endpoints

```bash
pip install fastapi uvicorn
```

```python
from fastapi import FastAPI

app = FastAPI()

@app.get('/')
def inicio():
    return {'mensaje': 'API de productos'}

@app.get('/productos/{producto_id}')
def obtener_producto(producto_id: int):
    return {'id': producto_id, 'nombre': 'Notebook Lenovo'}
```

FastAPI usa los **type hints** de Python (`producto_id: int`) para validar y convertir automáticamente los parámetros — si mandás algo que no es un número, devuelve un `422` sin que escribas ese código vos.

Documentación oficial: [Tutorial de FastAPI](https://fastapi.tiangolo.com/tutorial/)

In [1]:
!pip install -q fastapi uvicorn

from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get('/')
def inicio():
    return {'mensaje': 'API de productos'}

@app.get('/productos/{producto_id}')
def obtener_producto(producto_id: int):
    return {'id': producto_id, 'nombre': 'Notebook Lenovo'}

cliente = TestClient(app)
resp = cliente.get('/productos/1')
print(resp.status_code, resp.json())

resp = cliente.get('/productos/abc')
print(resp.status_code, resp.json())

200 {'id': 1, 'nombre': 'Notebook Lenovo'}
422 {'detail': [{'type': 'int_parsing', 'loc': ['path', 'producto_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'abc'}]}


### Ejercicio 1 — Endpoint con query params

Agregá a la `app` anterior un endpoint `GET /productos` que reciba dos query params opcionales: `categoria: str = None` y `limite: int = 10`, y devuelva `{'categoria': categoria, 'limite': limite}`. Probalo con `TestClient` pasando `params={'categoria': 'notebooks', 'limite': 5}`.

In [ ]:
# TODO: agregar el endpoint GET /productos con categoria y limite como query params

# TODO: probarlo con cliente.get('/productos', params={...})

<details>
<summary>💡 Ver solución</summary>

```python
from typing import Optional

@app.get('/productos')
def listar_productos(categoria: Optional[str] = None, limite: int = 10):
    return {'categoria': categoria, 'limite': limite}

resp = cliente.get('/productos', params={'categoria': 'notebooks', 'limite': 5})
print(resp.status_code, resp.json())
```

</details>

In [2]:
from typing import Optional

@app.get('/productos')
def listar_productos(categoria: Optional[str] = None, limite: int = 10):
    return {'categoria': categoria, 'limite': limite}

resp = cliente.get('/productos', params={'categoria': 'notebooks', 'limite': 5})
print(resp.status_code, resp.json())

200 {'categoria': 'notebooks', 'limite': 5}


## 3. Modelos con Pydantic

Pydantic define la forma de tus datos con clases Python normales, y valida automáticamente tipos, campos requeridos/opcionales y restricciones.

```python
from pydantic import BaseModel, Field

class ProductoIn(BaseModel):
    nombre: str
    precio: float = Field(gt=0)
    categoria: str = 'general'
    stock: int = 0
```

Si un cliente manda un precio negativo o se olvida el nombre, FastAPI devuelve automáticamente un `422` con el detalle del error, sin que escribas esa validación a mano.

Documentación oficial: [Pydantic](https://docs.pydantic.dev/latest/) · [Request Body en FastAPI](https://fastapi.tiangolo.com/tutorial/body/)

In [3]:
from pydantic import BaseModel, Field

class ProductoIn(BaseModel):
    nombre: str
    precio: float = Field(gt=0)
    categoria: str = 'general'
    stock: int = 0

In [4]:
from pydantic import BaseModel, Field

class ProductoIn(BaseModel):
    nombre: str
    precio: float = Field(gt=0)
    categoria: str = 'general'
    stock: int = 0

@app.post('/productos', status_code=201)
def crear_producto(producto: ProductoIn):
    return producto

cliente = TestClient(app)

resp = cliente.post('/productos', json={'nombre': 'Mouse', 'precio': 15.5})
print(resp.status_code, resp.json())

resp = cliente.post('/productos', json={'nombre': 'Mouse', 'precio': -5})
print(resp.status_code, resp.json())

201 {'nombre': 'Mouse', 'precio': 15.5, 'categoria': 'general', 'stock': 0}
422 {'detail': [{'type': 'greater_than', 'loc': ['body', 'precio'], 'msg': 'Input should be greater than 0', 'input': -5, 'ctx': {'gt': 0.0}}]}


### Ejercicio 2 — Validar con Pydantic

Agregá a `ProductoIn` un campo `sku: str` que sea obligatorio y tenga como mínimo 4 caracteres (usá `Field(min_length=4)`). Probá crear un producto sin `sku` y confirmá que devuelve `422`.

<details>
<summary>💡 Ver solución</summary>

```python
class ProductoIn(BaseModel):
    nombre: str
    precio: float = Field(gt=0)
    categoria: str = 'general'
    stock: int = 0
    sku: str = Field(min_length=4)

@app.post('/productos-v2', status_code=201)
def crear_producto_v2(producto: ProductoIn):
    return producto

resp = cliente.post('/productos-v2', json={'nombre': 'Mouse', 'precio': 15.5})
print(resp.status_code, resp.json())  # 422, falta sku
```

</details>

In [5]:
class ProductoIn(BaseModel):
    nombre: str
    precio: float = Field(gt=0)
    categoria: str = 'general'
    stock: int = 0
    sku: str = Field(min_length=4)

@app.post('/productos-v2', status_code=201)
def crear_producto_v2(producto: ProductoIn):
    return producto

resp = cliente.post('/productos-v2', json={'nombre': 'Mouse', 'precio': 15.5})
print(resp.status_code, resp.json())  # 422, falta sku

422 {'detail': [{'type': 'missing', 'loc': ['body', 'sku'], 'msg': 'Field required', 'input': {'nombre': 'Mouse', 'precio': 15.5}}]}


## 4. Serialización y `response_model`

Así como un modelo Pydantic valida lo que entra, otro modelo puede definir la forma exacta de lo que sale — útil para no exponer campos internos (por ejemplo, un costo interno que el cliente no debería ver).

```python
class ProductoOut(BaseModel):
    id: int
    nombre: str
    precio: float

@app.post('/productos-v3', response_model=ProductoOut, status_code=201)
def crear_producto_v3(producto: ProductoIn):
    return {**producto.model_dump(), 'id': 1, 'costo_interno': 8.0}  # costo_interno se descarta
```

FastAPI filtra automáticamente la respuesta según `response_model`, aunque la función devuelva campos de más.

In [6]:
class ProductoOut(BaseModel):
    id: int
    nombre: str
    precio: float

@app.post('/productos-v3', response_model=ProductoOut, status_code=201)
def crear_producto_v3(producto: ProductoIn):
    return {**producto.model_dump(), 'id': 1, 'costo_interno': 8.0}  # costo_interno se descarta

### Ejercicio 3 — `response_model`

Creá un endpoint `GET /productos/{producto_id}/resumen` que devuelva un diccionario con `id`, `nombre`, `precio` y también `margen_secreto` (un dato interno), pero usando `response_model=ProductoOut` para que `margen_secreto` no llegue nunca al cliente. Confirmá el resultado con `TestClient`.

<details>
<summary>💡 Ver solución</summary>

```python
@app.get('/productos/{producto_id}/resumen', response_model=ProductoOut)
def resumen_producto(producto_id: int):
    return {'id': producto_id, 'nombre': 'Mouse', 'precio': 15.5, 'margen_secreto': 40}

resp = cliente.get('/productos/1/resumen')
print(resp.status_code, resp.json())  # sin margen_secreto
```

</details>

In [7]:
@app.get('/productos/{producto_id}/resumen', response_model=ProductoOut)
def resumen_producto(producto_id: int):
    return {'id': producto_id, 'nombre': 'Mouse', 'precio': 15.5, 'margen_secreto': 40}

resp = cliente.get('/productos/1/resumen')
print(resp.status_code, resp.json())  # sin margen_secreto

200 {'id': 1, 'nombre': 'Mouse', 'precio': 15.5}


## 5. Documentación automática: Swagger UI y OpenAPI

FastAPI genera, sin configuración extra, un esquema **OpenAPI** a partir de tus rutas y modelos Pydantic, y lo expone en dos interfaces interactivas cuando corrés la API de verdad:

- `/docs` → **Swagger UI**: probar cada endpoint desde el navegador.
- `/redoc` → **ReDoc**: documentación de referencia, más orientada a lectura.

Podés enriquecer la documentación con metadata:

```python
app = FastAPI(title='API de Productos', version='1.0.0', description='Microservicio de catalogo')

@app.get('/productos/{producto_id}', summary='Obtener un producto por id', tags=['productos'])
def obtener_producto(producto_id: int):
    ...
```

Documentación oficial: [Metadata y docs en FastAPI](https://fastapi.tiangolo.com/tutorial/metadata/) · [Especificación OpenAPI](https://swagger.io/specification/) · [Swagger UI](https://swagger.io/tools/swagger-ui/)

In [8]:
from typing import Dict, List, Optional
from fastapi import FastAPI, HTTPException, Path, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

# =====================================================================
# 1. Esquemas de Datos (Pydantic)
# =====================================================================
class ProductoBase(BaseModel):
    nombre: str = Field(..., example="Teclado Mecánico RGB")
    categoria: str = Field(..., example="Periféricos")
    precio: float = Field(..., gt=0, description="El precio debe ser estrictamente mayor a 0", example=79.99)
    stock: int = Field(..., ge=0, description="El stock no puede ser negativo", example=25)


class ProductoResponse(ProductoBase):
    id: int = Field(..., example=1)


# =====================================================================
# 2. Inicialización de la Aplicación FastAPI
# =====================================================================
app = FastAPI(
    title="API de Productos",
    version="1.0.0",
    description="Microservicio de catálogo",
)

# Base de datos simulada en memoria
CATALOGO_DB: Dict[int, ProductoResponse] = {
    1: ProductoResponse(id=1, nombre="Notebook Lenovo ThinkPad", categoria="Computación", precio=1200.0, stock=8),
    2: ProductoResponse(id=2, nombre="Teclado Mecánico Keychron", categoria="Periféricos", precio=95.0, stock=15),
    3: ProductoResponse(id=3, nombre="Mouse Ergonómico MX Master", categoria="Periféricos", precio=110.0, stock=20),
}


# =====================================================================
# 3. Endpoints
# =====================================================================
@app.get(
    "/productos/{producto_id}",
    response_model=ProductoResponse,
    status_code=status.HTTP_200_OK,
    summary="Obtener un producto por id",
    tags=["productos"],
)
def obtener_producto(
    producto_id: int = Path(..., title="ID del producto", ge=1, description="Identificador único numérico mayor o igual a 1")
):
    """
    Recupera la información completa de un producto por su ID único:
    - **producto_id**: ID entero positivo del artículo.

    Retorna 404 Not Found si el ID no existe en el catálogo.
    """
    producto = CATALOGO_DB.get(producto_id)
    if not producto:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Producto con ID {producto_id} no encontrado en el catálogo.",
        )
    return producto


@app.get(
    "/productos",
    response_model=List[ProductoResponse],
    status_code=status.HTTP_200_OK,
    summary="Listar todos los productos",
    tags=["productos"],
)
def listar_productos(categoria: Optional[str] = None):
    """Lista todos los artículos disponibles con filtro opcional por categoría."""
    if categoria:
        return [p for p in CATALOGO_DB.values() if p.categoria.lower() == categoria.strip().lower()]
    return list(CATALOGO_DB.values())


# =====================================================================
# 4. Pruebas Automáticas con TestClient (Ejecutable en Colab / Script)
# =====================================================================
if __name__ == "__main__":
    cliente = TestClient(app)

    print("=" * 65)
    print("EJECUTANDO PRUEBAS DEL ENDPOINT /productos/{producto_id}")
    print("=" * 65)

    # Caso 1: Producto existente (ID 1)
    print("\n[Test 1] Consultar producto existente (ID = 1):")
    res1 = cliente.get("/productos/1")
    print(f"Status Code: {res1.status_code}")
    print("Respuesta:", res1.json())
    assert res1.status_code == 200
    assert res1.json()["nombre"] == "Notebook Lenovo ThinkPad"

    # Caso 2: Producto inexistente (ID 999) -> Debe responder 404
    print("\n[Test 2] Consultar producto inexistente (ID = 999):")
    res2 = cliente.get("/productos/999")
    print(f"Status Code: {res2.status_code}")
    print("Detalle de error:", res2.json()["detail"])
    assert res2.status_code == 404

    # Caso 3: ID con formato inválido (texto en vez de int) -> Debe responder 422
    print("\n[Test 3] Consultar con ID inválido ('abc'):")
    res3 = cliente.get("/productos/abc")
    print(f"Status Code: {res3.status_code}")
    assert res3.status_code == 422

    print("\n" + "=" * 65)
    print("✔ TODOS LOS TESTS DEL ENDPOINT PASARON CON ÉXITO")
    print("=" * 65)

EJECUTANDO PRUEBAS DEL ENDPOINT /productos/{producto_id}

[Test 1] Consultar producto existente (ID = 1):
Status Code: 200
Respuesta: {'nombre': 'Notebook Lenovo ThinkPad', 'categoria': 'Computación', 'precio': 1200.0, 'stock': 8, 'id': 1}

[Test 2] Consultar producto inexistente (ID = 999):
Status Code: 404
Detalle de error: Producto con ID 999 no encontrado en el catálogo.

[Test 3] Consultar con ID inválido ('abc'):
Status Code: 422

✔ TODOS LOS TESTS DEL ENDPOINT PASARON CON ÉXITO


/tmp/ipykernel_450/1901158815.py:10: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  nombre: str = Field(..., example="Teclado Mecánico RGB")
/tmp/ipykernel_450/1901158815.py:11: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  categoria: str = Field(..., example="Periféricos")
/tmp/ipykernel_450/1901158815.py:12: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be 

### Ejercicio 4 — Enriquecer la documentación

Agregále a la `app` un `title`, `version` y `description`, y al endpoint de creación de productos un `summary` y un `tags=['productos']`. (Este ejercicio se valida mirando `/docs` en el navegador cuando corrés la API con `uvicorn`, más adelante en este notebook.)

<details>
<summary>💡 Ver solución</summary>

```python
app = FastAPI(
    title='API de Productos',
    version='1.0.0',
    description='Microservicio de catalogo de productos',
)

@app.post('/productos', status_code=201, summary='Crear un producto', tags=['productos'])
def crear_producto(producto: ProductoIn):
    return producto
```

</details>

In [9]:
app = FastAPI(
    title='API de Productos',
    version='1.0.0',
    description='Microservicio de catalogo de productos',
)

@app.post('/productos', status_code=201, summary='Crear un producto', tags=['productos'])
def crear_producto(producto: ProductoIn):
    return producto

## 6. Correr la API de verdad

`TestClient` es perfecto para probar dentro del notebook, pero para ver `/docs` en el navegador necesitás un servidor real:

```bash
uvicorn main:app --reload
```

Esto la deja disponible en `http://127.0.0.1:8000/docs`. En Colab, para exponerla públicamente y poder abrir esa URL, se suele usar un túnel (por ejemplo con [pyngrok](https://pyngrok.readthedocs.io/en/latest/)) — algo opcional y fuera del alcance de este notebook, pero útil si querés mostrar la documentación interactiva en vivo.

## 7. El mismo endpoint en Flask (comparación)

```python
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route('/productos', methods=['POST'])
def crear_producto():
    data = request.get_json()
    if not data.get('nombre') or data.get('precio', 0) <= 0:
        return jsonify({'error': 'datos invalidos'}), 422
    return jsonify(data), 201
```

En Flask, la validación (el `if` de arriba) y la documentación quedan a tu cargo o requieren extensiones adicionales — es el trade-off frente a la ergonomía automática de FastAPI + Pydantic.

In [10]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route('/productos', methods=['POST'])
def crear_producto():
    data = request.get_json()
    if not data.get('nombre') or data.get('precio', 0) <= 0:
        return jsonify({'error': 'datos invalidos'}), 422
    return jsonify(data), 201

## Mini-proyecto: CRUD de tareas

Construí, en FastAPI, un CRUD completo en memoria (una lista de Python como base de datos) para un recurso `tarea` con: `id`, `titulo`, `completada: bool = False`.

- `POST /tareas` — crear (Pydantic valida que `titulo` no esté vacío)
- `GET /tareas` — listar, con query param opcional `completada`
- `GET /tareas/{id}` — obtener una, `404` si no existe
- `PUT /tareas/{id}` — reemplazar
- `DELETE /tareas/{id}` — borrar

Probá los 5 endpoints con `TestClient`.

**Entregable:** el código de la API + las pruebas con `TestClient` mostrando cada caso (incluido el `404`).

---

**Seguís en:** *Colab 3 — Microservicio de predicciones y métricas de negocio*

In [11]:
from typing import Dict, List, Optional
from fastapi import FastAPI, HTTPException, Path, Query, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, field_validator

# =====================================================================
# 1. Esquemas de Datos con Pydantic
# =====================================================================
class TareaCreate(BaseModel):
    titulo: str = Field(..., description="Título descriptivo de la tarea")
    completada: bool = Field(default=False, description="Estado de realización")

    @field_validator("titulo")
    @classmethod
    def validar_titulo_no_vacio(cls, valor: str) -> str:
        if not valor or not valor.strip():
            raise ValueError("El título no puede estar vacío ni contener solo espacios.")
        return valor.strip()


class TareaUpdate(BaseModel):
    titulo: str = Field(..., description="Nuevo título de la tarea")
    completada: bool = Field(..., description="Nuevo estado de la tarea")

    @field_validator("titulo")
    @classmethod
    def validar_titulo_no_vacio(cls, valor: str) -> str:
        if not valor or not valor.strip():
            raise ValueError("El título no puede estar vacío ni contener solo espacios.")
        return valor.strip()


class TareaResponse(BaseModel):
    id: int
    titulo: str
    completada: bool


# =====================================================================
# 2. Aplicación FastAPI y Estado en Memoria
# =====================================================================
app = FastAPI(
    title="API de Tareas",
    description="Microservicio CRUD en memoria para gestión de tareas",
    version="1.0.0",
)

# Base de datos en memoria (lista de diccionarios) y contador de autoincremento
db_tareas: List[Dict] = []
secuencia_id = 0


# =====================================================================
# 3. Endpoints del CRUD
# =====================================================================

# POST /tareas: Crear
@app.post(
    "/tareas",
    response_model=TareaResponse,
    status_code=status.HTTP_201_CREATED,
    summary="Crear una nueva tarea",
    tags=["tareas"],
)
def crear_tarea(tarea: TareaCreate):
    global secuencia_id
    secuencia_id += 1
    nueva_tarea = {
        "id": secuencia_id,
        "titulo": tarea.titulo,
        "completada": tarea.completada,
    }
    db_tareas.append(nueva_tarea)
    return nueva_tarea


# GET /tareas: Listar (con filtro opcional completada)
@app.get(
    "/tareas",
    response_model=List[TareaResponse],
    status_code=status.HTTP_200_OK,
    summary="Listar tareas",
    tags=["tareas"],
)
def listar_tareas(completada: Optional[bool] = Query(None, description="Filtrar por estado booleano")):
    if completada is not None:
        return [t for t in db_tareas if t["completada"] == completada]
    return list(db_tareas)


# GET /tareas/{id}: Obtener una por ID (404 si no existe)
@app.get(
    "/tareas/{tarea_id}",
    response_model=TareaResponse,
    status_code=status.HTTP_200_OK,
    summary="Obtener una tarea por su ID",
    tags=["tareas"],
)
def obtener_tarea(
    tarea_id: int = Path(..., ge=1, description="ID numérico de la tarea")
):
    for t in db_tareas:
        if t["id"] == tarea_id:
            return t
    raise HTTPException(
        status_code=status.HTTP_404_NOT_FOUND,
        detail=f"Tarea con ID {tarea_id} no encontrada.",
    )


# PUT /tareas/{id}: Reemplazar tarea completa
@app.put(
    "/tareas/{tarea_id}",
    response_model=TareaResponse,
    status_code=status.HTTP_200_OK,
    summary="Reemplazar una tarea existente",
    tags=["tareas"],
)
def reemplazar_tarea(
    tarea_actualizada: TareaUpdate,
    tarea_id: int = Path(..., ge=1, description="ID numérico de la tarea"),
):
    for i, t in enumerate(db_tareas):
        if t["id"] == tarea_id:
            tarea_modificada = {
                "id": tarea_id,
                "titulo": tarea_actualizada.titulo,
                "completada": tarea_actualizada.completada,
            }
            db_tareas[i] = tarea_modificada
            return tarea_modificada

    raise HTTPException(
        status_code=status.HTTP_404_NOT_FOUND,
        detail=f"No se pudo reemplazar: Tarea con ID {tarea_id} no encontrada.",
    )


# DELETE /tareas/{id}: Borrar tarea
@app.delete(
    "/tareas/{tarea_id}",
    status_code=status.HTTP_200_OK,
    summary="Borrar una tarea",
    tags=["tareas"],
)
def borrar_tarea(
    tarea_id: int = Path(..., ge=1, description="ID numérico de la tarea")
):
    for i, t in enumerate(db_tareas):
        if t["id"] == tarea_id:
            tarea_eliminada = db_tareas.pop(i)
            return {
                "mensaje": f"Tarea {tarea_id} eliminada exitosamente.",
                "tarea": tarea_eliminada,
            }

    raise HTTPException(
        status_code=status.HTTP_404_NOT_FOUND,
        detail=f"No se pudo eliminar: Tarea con ID {tarea_id} no encontrada.",
    )


# =====================================================================
# 4. Pruebas Automatizadas con TestClient
# =====================================================================
if __name__ == "__main__":
    cliente = TestClient(app)

    print("=" * 70)
    print("EJECUTANDO BATERÍA DE PRUEBAS PARA EL CRUD DE TAREAS")
    print("=" * 70)

    # 1. POST /tareas (Creación válida)
    print("\n[1] POST /tareas (Crear tareas válidas):")
    r1 = cliente.post("/tareas", json={"titulo": "Diseñar arquitectura"})
    r2 = cliente.post("/tareas", json={"titulo": "Configurar CI/CD", "completada": True})
    r3 = cliente.post("/tareas", json={"titulo": "Escribir tests unitarios"})
    print(f"Creada 1 -> Status: {r1.status_code} | Data: {r1.json()}")
    print(f"Creada 2 -> Status: {r2.status_code} | Data: {r2.json()}")
    assert r1.status_code == 201 and r1.json()["id"] == 1
    assert r2.status_code == 201 and r2.json()["completada"] is True

    # 2. POST /tareas (Validación: Título vacío o con espacios)
    print("\n[2] POST /tareas (Validación: rechazar título vacío):")
    r_vacio = cliente.post("/tareas", json={"titulo": "   "})
    print(f"Status: {r_vacio.status_code} (Esperado: 422)")
    assert r_vacio.status_code == 422

    # 3. GET /tareas (Listado sin filtro)
    print("\n[3] GET /tareas (Listado completo):")
    r_list = cliente.get("/tareas")
    print(f"Total registradas: {len(r_list.json())} tareas.")
    assert len(r_list.json()) == 3

    # 4. GET /tareas?completada=true (Filtro por query param)
    print("\n[4] GET /tareas?completada=true (Filtrar solo completadas):")
    r_filtro = cliente.get("/tareas?completada=true")
    print(f"Status: {r_filtro.status_code} | Tareas completadas: {r_filtro.json()}")
    assert len(r_filtro.json()) == 1
    assert r_filtro.json()[0]["id"] == 2

    # 5. GET /tareas/{id} (Existente y 404 Inexistente)
    print("\n[5] GET /tareas/{id} (Existente vs 404):")
    r_id_ok = cliente.get("/tareas/1")
    r_id_404 = cliente.get("/tareas/999")
    print(f"ID 1 -> Status: {r_id_ok.status_code} | Titulo: {r_id_ok.json()['titulo']}")
    print(f"ID 999 -> Status: {r_id_404.status_code} | Detalle: {r_id_404.json()['detail']}")
    assert r_id_ok.status_code == 200
    assert r_id_404.status_code == 404

    # 6. PUT /tareas/{id} (Reemplazo exitoso y 404)
    print("\n[6] PUT /tareas/{id} (Actualizar tarea existente y 404):")
    payload_update = {"titulo": "Diseñar arquitectura v2 (Aprobada)", "completada": True}
    r_put_ok = cliente.put("/tareas/1", json=payload_update)
    r_put_404 = cliente.put("/tareas/999", json=payload_update)
    print(f"PUT ID 1 -> Status: {r_put_ok.status_code} | Resultado: {r_put_ok.json()}")
    print(f"PUT ID 999 -> Status: {r_put_404.status_code} | Detalle: {r_put_404.json()['detail']}")
    assert r_put_ok.status_code == 200 and r_put_ok.json()["completada"] is True
    assert r_put_404.status_code == 404

    # 7. DELETE /tareas/{id} (Borrado exitoso y confirmación 404)
    print("\n[7] DELETE /tareas/{id} (Borrado y verificación posterior):")
    r_del = cliente.delete("/tareas/1")
    print(f"DELETE ID 1 -> Status: {r_del.status_code} | Mensaje: {r_del.json()['mensaje']}")
    assert r_del.status_code == 200

    # Comprobar que ya no existe (404)
    r_del_check = cliente.get("/tareas/1")
    print(f"GET ID 1 tras borrado -> Status: {r_del_check.status_code} (Esperado: 404)")
    assert r_del_check.status_code == 404

    # Intento de borrar ID inexistente
    r_del_404 = cliente.delete("/tareas/999")
    print(f"DELETE ID 999 -> Status: {r_del_404.status_code} | Detalle: {r_del_404.json()['detail']}")
    assert r_del_404.status_code == 404

    print("\n" + "=" * 70)
    print("✔ TODOS LOS ENDPOINTS Y CASOS DE ERROR (404/422) FUERON VALIDADOS")
    print("=" * 70)

EJECUTANDO BATERÍA DE PRUEBAS PARA EL CRUD DE TAREAS

[1] POST /tareas (Crear tareas válidas):
Creada 1 -> Status: 201 | Data: {'id': 1, 'titulo': 'Diseñar arquitectura', 'completada': False}
Creada 2 -> Status: 201 | Data: {'id': 2, 'titulo': 'Configurar CI/CD', 'completada': True}

[2] POST /tareas (Validación: rechazar título vacío):
Status: 422 (Esperado: 422)

[3] GET /tareas (Listado completo):
Total registradas: 3 tareas.

[4] GET /tareas?completada=true (Filtrar solo completadas):
Status: 200 | Tareas completadas: [{'id': 2, 'titulo': 'Configurar CI/CD', 'completada': True}]

[5] GET /tareas/{id} (Existente vs 404):
ID 1 -> Status: 200 | Titulo: Diseñar arquitectura
ID 999 -> Status: 404 | Detalle: Tarea con ID 999 no encontrada.

[6] PUT /tareas/{id} (Actualizar tarea existente y 404):
PUT ID 1 -> Status: 200 | Resultado: {'id': 1, 'titulo': 'Diseñar arquitectura v2 (Aprobada)', 'completada': True}
PUT ID 999 -> Status: 404 | Detalle: No se pudo reemplazar: Tarea con ID 999 no